# `LG_Graphics` tutorial

Thin PyVista wrapper in `brownian_ot.graphics` for drawing brownian_ot particles
and making movies from ASDF trajectories.

**Covers**
1. Init options (`off_screen`, `add_lab_axes`)
2. Static draws: `Sphere`, `Spheroid`, `Dimer`, `SphereCluster`
3. Lab / particle axes and LG-beam chrome
4. Movies: `movie`, `movie_lab`, in-memory `asdf_trajectory`, manual loop

**ASDF (Jerome schema)**
- Trajectory `(n, 7)` → `x, y, z, qw, qx, qy, qz` (row 0 = initial pose)
- Particles: `Sphere` (`radius`), `Spheroid` (`perpendicular_radius`, `aspect_ratio`),
  `Dimer` / `SphereCluster` (`radius`, `sphere_positions`, optional `aspect_ratios`)

Requires: `pyvista`, `asdf`, `imageio-ffmpeg` (mp4), OpenGL / OSMesa for off-screen.


## 0. Setup


In [ ]:
from pathlib import Path
import sys

import numpy as np
import asdf
import pyvista as pv

# Repo layout: .../lg_beam/brownian_ot/examples/this_notebook.ipynb
REPO = Path("../..").resolve()  # lg_beam
PKG = Path("..").resolve()      # brownian_ot package root
sys.path.insert(0, str(PKG))

from brownian_ot.graphics import LG_Graphics

# Static images in the notebook (no interactive WebGL widget)
pv.set_jupyter_backend("static")

OUT = Path("graphics_demo_out")
OUT.mkdir(exist_ok=True)

TEXTURE_PATH = REPO / "cole-and-son-polo-stripe-leaf-green-wallpaper-tiled-142761.jpg"
# tex = pv.Texture(str(TEXTURE_PATH)) if TEXTURE_PATH.is_file() else None
tex = None  # set to a Texture for wallpaper demos; None is faster

print("texture:", TEXTURE_PATH.name if tex else "(none)")
print("output:", OUT.resolve())


## 1. Init options

```python
LG_Graphics()                              # on-screen plotter + lab axes (static demos)
LG_Graphics(add_lab_axes=False)            # bare plotter (or before movies)
LG_Graphics(off_screen=True, add_lab_axes=False)  # headless / WSL-friendly starter
```

`movie` / `movie_lab` **replace** `self.pl` with their own off-screen plotter, so for
movies prefer `add_lab_axes=False` to skip building an unused starter scene.


In [ ]:
# Static demo: default init draws lab arrows automatically.
g_axes = LG_Graphics()
g_axes.show()


## 2. Sphere

Single sphere: `add_sphere(a, pos, quat, texture=..., key=...)`.

Quaternions are OtSim / numpy-quaternion order: **`(w, qx, qy, qz)`**.


In [ ]:
g_sph = LG_Graphics()
g_sph.setRotationFromEuler([15, 40, -10], degrees=True)
g_sph.add_sphere(0.5, np.zeros(3), g_sph.quat, texture=tex, key="sphere")
g_sph.add_particle_axes(np.zeros(3), g_sph.quat, length=1.8)
g_sph.show()


## 3. Spheroid

`Spheroid` is one ellipsoid of revolution (not a cluster). Body **+z** is the symmetry axis.

| arg / ASDF | meaning |
|---|---|
| `a` / `perpendicular_radius` | equatorial radius |
| `ar` / `aspect_ratio` | polar radius = `a * ar` (`ar > 1` prolate, `ar < 1` oblate) |

There is **no** `SpheroidCluster` in brownian_ot — only `SphereCluster` (spheres).


In [ ]:
g_elo = LG_Graphics()
g_elo.setRotationFromEuler([25, -20, 15], degrees=True)

# Prolate (cigar) and oblate (pancake) side by side
g_elo.add_spheroid(0.35, 2.0, np.array([-1.2, 0.0, 0.0]), g_elo.quat, texture=tex, key="prolate")
g_elo.add_spheroid(0.55, 0.4, np.array([1.2, 0.0, 0.0]), g_elo.quat, texture=tex, key="oblate")
g_elo.add_particle_axes(np.array([-1.2, 0.0, 0.0]), g_elo.quat, length=1.6)
g_elo.show()


## 4. Dimer + particle axes

`add_dimer` places two equal spheres along body ±z. `add_particle_axes` draws the
body triad that follows the quaternion (unlike fixed lab arrows).


In [ ]:
g_dim = LG_Graphics()
g_dim.setRotationFromEuler([20, 35, 10], degrees=True)
g_dim.add_dimer(0.5, np.zeros(3), g_dim.quat, texture=tex, key="dimer")
g_dim.add_particle_axes(np.zeros(3), g_dim.quat, length=2.0)
g_dim.show()


## 5. Sphere cluster (asymmetric trimer)

There is no `Trimer` type — trimers are `SphereCluster`.
`add_sphere_cluster(a, pos, a_pos, quat, a_ratio)` matches ASDF:

| arg | meaning | ASDF |
|---|---|---|
| `a` | dimensional radius scale | `particle["radius"]` |
| `pos` | lab COM | `trajectory[i, :3]` |
| `a_pos` | body centers in units of `a` | `particle["sphere_positions"]` |
| `a_ratio` | per-sphere size ratios | `particle["aspect_ratios"]` |


In [ ]:
c = 0.25  # a_small / a_big
xcm = LG_Graphics.xcm_nondim(c)
s = np.sqrt(c**2 + 2 * c)

a_pos = np.array(
    [
        [-xcm, 0.0, 1.0],     # big
        [-xcm, 0.0, -1.0],    # big
        [s - xcm, 0.0, 0.0],  # small
    ]
)
a_ratio = np.array([1.0, 1.0, c])

g_tri = LG_Graphics()
g_tri.setRotationFromEuler([10, -25, 40], degrees=True)
g_tri.add_sphere_cluster(0.5, np.zeros(3), a_pos, g_tri.quat, a_ratio, texture=tex, key="tri")
g_tri.add_particle_axes(np.zeros(3), g_tri.quat, length=2.2)
g_tri.show()


## 6. LG beam chrome

Beam drawing is independent of the particle — add it when you want trap context.


In [ ]:
g_beam = LG_Graphics()
g_beam.add_lg_beam(w0=0.6, zR=1.6, p=0, ell=2, isosurface=0.10, show_envelope=True)
g_beam.setRotationFromEuler([0, 0, 0], degrees=True)
g_beam.add_dimer(0.4, np.array([0.8, 0.0, 0.2]), g_beam.quat, texture=tex, key="dimer")
g_beam.show()


## 7. Peek at a dimer ASDF file

**Data used for movies below**

| | |
|---|---|
| File | `lg_simulations/data/dimer_modes/D2_LCP_0.03_5447152_1ms.asdf` |
| Particle | `Dimer`, `radius = 0.4 µm` |
| This file | **30 001** rows at 1 ms → **30 s** of physics |

Movie length is *not* the simulation length:

\[
t_\mathrm{movie} \approx \frac{N_\mathrm{rows}/\texttt{slowdown}}{\texttt{framerate}}
\]

`slowdown` = stride. `n_frames` = max **written** mp4 frames after striding.


In [ ]:
ASDF_FILE = REPO / "lg_simulations/data/dimer_modes/D2_LCP_0.03_5447152_1ms.asdf"
assert ASDF_FILE.is_file(), f"missing {ASDF_FILE}"

with asdf.open(ASDF_FILE) as af:
    print("top-level:", [k for k in af.tree if k not in ("asdf_library", "history")])
    print("particle:", dict(af["particle"]))
    print("simulation:", dict(af["simulation"]))
    traj = np.asarray(af["trajectory"])
    sim = af["simulation"]
    dt_phys = float(sim.get("downsampled_timestep", sim["timestep"]))

print("trajectory shape:", traj.shape)
print("initial pose (row 0):", traj[0])
print("pos range [m]:", traj[:, :3].min(0), "→", traj[:, :3].max(0))
print(f"physics duration: {(traj.shape[0] - 1) * dt_phys:.1f} s")

FRAMERATE = 24
TARGET_SEC = 2  # raise for longer videos (e.g. 12.5)
SLOWDOWN = max(1, int(round(traj.shape[0] / (TARGET_SEC * FRAMERATE))))
N_WRITTEN = int(np.ceil(traj.shape[0] / SLOWDOWN))
print(
    f"movie settings: slowdown={SLOWDOWN}, framerate={FRAMERATE} "
    f"→ ~{N_WRITTEN} frames → ~{N_WRITTEN / FRAMERATE:.1f} s video"
)


## 8. Single-pane movie (`movie`)

ASDF lengths are SI meters; use `length_scale=1e6` (m → µm).

- Omit `n_frames` to stride the whole file.
- Smoke test: `n_frames=48` writes 48 mp4 frames after `slowdown`.


In [ ]:
g_movie = LG_Graphics(add_lab_axes=False)  # movies replace self.pl anyway

g_movie.movie(
    asdf_file=str(ASDF_FILE),
    name=str(OUT / "dimer_lab"),
    length_scale=1e6,
    slowdown=SLOWDOWN,
    # n_frames=48,          # uncomment for a quick smoke test
    framerate=FRAMERATE,
    quality=5,
    texture=tex,
    off_screen=True,
)
print("wrote", (OUT / "dimer_lab.mp4").resolve())
print(f"expected length ≈ {N_WRITTEN / FRAMERATE:.1f} s")


## 9. Side-by-side movie (`movie_lab`)

PyVista `shape=(1, 2)`:

- **Left:** lab motion inside a bounds box (`pos` + `quat`)
- **Right:** same orientation, COM at origin + lab arrows


In [ ]:
g_side = LG_Graphics(add_lab_axes=False)

g_side.movie_lab(
    asdf_file=str(ASDF_FILE),
    name=str(OUT / "dimer_side_by_side"),
    length_scale=1e6,
    slowdown=SLOWDOWN,
    framerate=FRAMERATE,
    quality=5,
    texture=tex,
    window_size=(1600, 800),
    off_screen=True,
)
print("wrote", (OUT / "dimer_side_by_side.mp4").resolve())
print(f"expected length ≈ {N_WRITTEN / FRAMERATE:.1f} s")


## 10. In-memory tree (`asdf_trajectory=...`)

Pass a dict with `trajectory` + `particle` instead of a file path.
Useful for synthetic demos (e.g. a spinning spheroid) without writing ASDF.


In [ ]:
# Synthetic spinning spheroid: fixed COM, slowly rotating about lab z
n = 120
t = np.linspace(0, 2 * np.pi, n, endpoint=False)
traj_syn = np.zeros((n, 7))
traj_syn[:, 3] = np.cos(t / 2)  # qw
traj_syn[:, 6] = np.sin(t / 2)  # qz  (rotation about +z)

tree = {
    "trajectory": traj_syn,
    "particle": {
        "type": "Spheroid",
        "perpendicular_radius": 0.4e-6,
        "aspect_ratio": 2.0,
        "center_of_diffusion": np.zeros(3),
    },
}

g_syn = LG_Graphics(add_lab_axes=False)
g_syn.movie(
    asdf_trajectory=tree,
    name=str(OUT / "spheroid_spin"),
    length_scale=1e6,
    slowdown=1,
    framerate=24,
    quality=5,
    texture=tex,
    off_screen=True,
)
print("wrote", (OUT / "spheroid_spin.mp4").resolve())


## 11. Manual frame loop

Use when you want a custom scene (beam, extra labels, …).
Reuse the same actor `key` each frame — meshes update via transform (no remesh).


In [ ]:
with asdf.open(ASDF_FILE) as data:
    traj = np.asarray(data["trajectory"])[::SLOWDOWN].copy()
    part = dict(data["particle"])

traj[:, :3] *= 1e6
part["radius"] = float(part["radius"]) * 1e6

g_custom = LG_Graphics(off_screen=True, add_lab_axes=False)
g_custom.LabAxis_font_size = 28
g_custom.LabAxis_axisLength = 3
g_custom._setup_lab_axes()
g_custom.add_lg_beam(w0=0.8, zR=2.0, ell=2, p=0, show_envelope=False)
g_custom.pl.camera_position = g_custom._default_camera()

out = OUT / "dimer_with_beam.mp4"
g_custom.pl.open_movie(str(out), framerate=FRAMERATE, quality=5)
for row in traj:
    g_custom._draw_particle(part, row[:3], row[3:], key="lab", texture=tex)
    g_custom.pl.write_frame()
g_custom.pl.close()
print("wrote", out.resolve())
print(f"expected length ≈ {len(traj) / FRAMERATE:.1f} s")


## Cheat sheet

```text
LG_Graphics
├── __init__(off_screen=False, add_lab_axes=True)
├── add_sphere / add_spheroid / add_dimer / add_sphere_cluster
├── add_particle_axes / add_axis_lab / add_lg_beam
├── movie(...)        # one pane, lab motion
└── movie_lab(...)    # left lab box | right orientation at origin
```

| want | call / ASDF |
|---|---|
| Sphere | `add_sphere` / `type=="Sphere"` + `radius` |
| Spheroid | `add_spheroid(a, ar, ...)` / `perpendicular_radius` + `aspect_ratio` |
| Dimer | `add_dimer` or cluster path / `type=="Dimer"` + `sphere_positions` |
| Trimer / cluster | `add_sphere_cluster` / `SphereCluster` + positions + ratios |
| SI → µm | `length_scale=1e6` |
| Longer video | smaller `slowdown` or lower `framerate` |
| Smoke test | `n_frames=48` (written frames after stride) |
| No file I/O | `asdf_trajectory={trajectory, particle}` |

Movie length: \(t \approx (N_\mathrm{rows}/\texttt{slowdown})/\texttt{framerate}\)
(capped by `n_frames` if set).

Initial pose is always `trajectory[0]` — no separate `pos0` / `orient0` keys.
